<center><h1>Last_First_HW3</h1></center>
<br>
<br>

Name: 
<br>
Github Username: 
<br>
USC ID: 

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [19]:
from pathlib import Path
from zipfile import ZipFile
import numpy as np
import pandas as pd

Get the AReM Data Set

In [31]:
data_dir = Path("AReM")
csv_files = sorted(data_dir.rglob("*.csv"))

print(len(csv_files))

88


### (b) Test and Train Data

In [34]:
test_files = []
train_files = []

for file in csv_files:
    activity = file.parent.name.lower()
    dataset_num = int(file.stem.replace("dataset", ""))

    test_max = 2 if activity in {"bending1", "bending2"} else 3

    if dataset_num <= test_max:
        test_files.append(file)
    else:
        train_files.append(file)

print("Training files:", len(train_files))
print("Test files:", len(test_files))

Training files: 69
Test files: 19


### (c) Feature Extraction

#### i. Research

Common time-domain features include minimum, maximum, mean, median, standard deviation, variance, range, quartiles, root mean square, skewness, and kurtosis.

#### ii. Extraction

In [60]:
columns = [
    "time", "avg_rss12", "var_rss12",
    "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"
]
sensor_columns = columns[1:]

def extract_features(file):
    rows = []

    for line_number, line in enumerate(file.read_text().splitlines()[5:], start=6):
        values = line.replace(",", " ").split()

        if not values:
            continue

        if len(values) != 7:
            raise ValueError(
                f"{file}, line {line_number}: expected 7 values, found {len(values)}"
            )

        rows.append(values)

    data = pd.DataFrame(rows, columns=columns, dtype=float)
    row = {}

    for col in sensor_columns:
        s = data[col]
        row[f"{col}_min"] = s.min()
        row[f"{col}_max"] = s.max()
        row[f"{col}_mean"] = s.mean()
        row[f"{col}_median"] = s.median()
        row[f"{col}_std"] = s.std()
        row[f"{col}_first_quartile"] = s.quantile(0.25)
        row[f"{col}_third_quartile"] = s.quantile(0.75)

    row["activity"] = (
        "bending" if file.parent.name.lower().startswith("bending")
        else file.parent.name.lower()
    )
    row["file"] = str(file.relative_to(data_dir))
    return row

features_df = pd.DataFrame([extract_features(file) for file in csv_files])
features_df.head()

,avg_rss12_min,avg_rss12_max,avg_rss12_mean,avg_rss12_median,avg_rss12_std,avg_rss12_first_quartile,avg_rss12_third_quartile,var_rss12_min,var_rss12_max,var_rss12_mean,...,avg_rss23_third_quartile,var_rss23_min,var_rss23_max,var_rss23_mean,var_rss23_median,var_rss23_std,var_rss23_first_quartile,var_rss23_third_quartile,activity,file
0,37.25,45.00,40.624792,40.50,1.476967,39.25,42.00,0.0,1.30,0.358604,...,36.00,0.0,1.92,0.570583,0.43,0.582915,0.0,1.30,bending,bending1/dataset1.csv
1,38.00,45.67,42.812812,42.50,1.435550,42.00,43.67,0.0,1.22,0.372437,...,34.50,0.0,3.11,0.571083,0.43,0.601010,0.0,1.30,bending,bending1/dataset2.csv
2,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,0.0,1.70,0.426250,...,36.50,0.0,1.79,0.493292,0.43,0.513506,0.0,0.94,bending,bending1/dataset3.csv
3,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,0.0,3.00,0.696042,...,36.33,0.0,2.18,0.613521,0.50,0.524317,0.0,1.00,bending,bending1/dataset4.csv
4,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,0.0,2.83,0.535979,...,31.25,0.0,1.79,0.383292,0.43,0.389164,0.0,0.50,bending,bending1/dataset5.csv


#### iii. Standard Deviation

In [56]:
feature_cols = [
    col for col in features_df.columns
    if col not in ["activity", "file"]
]

values = features_df[feature_cols].to_numpy()
estimated_sds = np.std(values, axis=0, ddof=1)

rng = np.random.default_rng(1)
bootstrap_sds = np.empty((1000, len(feature_cols)))

for b in range(1000):
    indices = rng.integers(0, len(values), size=len(values))
    bootstrap_sds[b] = np.std(values[indices], axis=0, ddof=1)

ci_lower, ci_upper = np.percentile(bootstrap_sds, [5, 95], axis=0)

std_results = pd.DataFrame({
    "Feature": feature_cols,
    "Estimated SD": estimated_sds,
    "90% CI lower": ci_lower,
    "90% CI upper": ci_upper
})

std_results

,Feature,Estimated SD,90% CI lower,90% CI upper
0,avg_rss12_min,9.569975,8.278846,10.908792
1,avg_rss12_max,4.394362,3.287317,5.292062
2,avg_rss12_mean,5.335718,4.701487,5.889290
3,avg_rss12_median,5.440054,4.796361,6.023666
4,avg_rss12_std,1.772153,1.571117,1.948094
5,avg_rss12_first_quartile,6.153590,5.579874,6.662732
6,avg_rss12_third_quartile,5.138925,4.313154,5.826573
7,var_rss12_min,0.000000,0.000000,0.000000
8,var_rss12_max,5.062729,4.610176,5.402727
9,var_rss12_mean,1.574164,1.397591,1.708244


#### iv. Select Features

I selected the minimum, mean, and maximum as the three most important time-domain features. The mean describes the typical sensor reading, while the minimum and maximum capture its lower and upper values. Together, they give a simple summary of each time series.

## 2. ISLR 3.7.4

### (a) Linear Train

The cubic regression will have training RSS less than or equal to the linear regression’s. The linear model is a special case of the cubic model, so adding the quadratic and cubic terms cannot make the training fit worse.

### (b) Linear Test

We expect the linear regression to have lower test RSS on average because the true relationship is linear. The cubic model’s extra terms can fit noise in the training data and make its predictions less accurate.

### (c) Not Linear Train

The cubic regression will again have training RSS less than or equal to the linear regression’s. This is true whether or not the true relationship is linear, because the cubic model includes the linear model as a special case.

### (d) Not Linear Testing

There is not enough information to determine which model will have lower test RSS. It depends on how the true relationship differs from a straight line: a cubic model may do better if it captures that pattern, while a linear model may do better if the cubic terms mostly fit noise.

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 